Batch testing

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the plate data
plates = pd.read_pickle('R:/toshiba usb/merged_p1_30k.pkl')
plates2 = pd.read_pickle('R:/toshiba usb/merged_p2_30k.pkl')
plates3 = pd.read_pickle('R:/toshiba usb/merged_p3_30k.pkl')

# Function to generate heatmap for a given plate
def generate_heatmap(plate_data, plate_name):
    # Initialize lists to store the extracted data
    appended_data = []
    appended_data_letter = []
    appended_data_number = []
    appended_data2 = []

    # List of letters and numbers representing rows and columns of the 96-well plate
    letter = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
    num = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

    # Loop through wells by their letter and number, extracting and appending data
    for i in range(len(num)):
        r = str(num[i])

        for x in range(len(letter)):
            well = str(letter[x]) + str(num[i])

            try:
                # Count occurrences of each well in the 'Display Name' column
                well_count = plate_data[plate_data['Display Name'] == well].shape[0]

                # Append the well data and counts
                appended_data.append(well)
                appended_data_letter.append(str(letter[x]))
                appended_data_number.append(str(num[i]))
                appended_data2.append(well_count)

            except:
                # If an error occurs, append the well name and set count as 0
                appended_data.append([well])
                appended_data_letter.append(str(letter[x]))
                appended_data_number.append(str(num[i]))
                appended_data2.append(0)

    # Create a DataFrame from the appended lists
    dfx1 = pd.DataFrame({'wells': appended_data})
    dfx2 = pd.DataFrame({'letter': appended_data_letter})
    dfx3 = pd.DataFrame({'number': appended_data_number})
    dfx4 = pd.DataFrame({'count': appended_data2})

    # Concatenate the data into a single DataFrame
    df_final = pd.concat([dfx1, dfx2, dfx3, dfx4], axis=1)

    # Pivot the data to format for a heatmap (row: 'letter', column: 'number', values: 'count')
    df_heatmap = df_final.pivot(index='letter', columns='number', values='count')

    # Plot the heatmap
    plt.figure(figsize=(10, 6))  # Slightly smaller figure size
    sns.heatmap(
        df_heatmap,
        cmap="YlGnBu",
        annot=True,
        fmt=".0f",
        cbar_kws={'label': 'Cell Counts'},
        annot_kws={'size': 10},  # Increase annotation font size
    )
    plt.title(f'{plate_name} 96-Well Plate Cell Count Heatmap', fontsize=16)  # Larger title font
    plt.xlabel('Column Number', fontsize=14)  # Larger x-axis label font
    plt.ylabel('Row Letter', fontsize=14)  # Larger y-axis label font
    plt.xticks(fontsize=12)  # Larger x-axis ticks font
    plt.yticks(fontsize=12)  # Larger y-axis ticks font
    plt.show()

# Modify Plate 3 to switch counts for wells E02 and F02
def switch_wells(plate_data):
    # Find counts for wells E02 and F02
    E02_count = plate_data[plate_data['Display Name'] == 'E02'].shape[0]
    F02_count = plate_data[plate_data['Display Name'] == 'F02'].shape[0]
    
    # Switch the values in the dataset
    plate_data.loc[plate_data['Display Name'] == 'E02', 'Display Name'] = 'F02_temp'
    plate_data.loc[plate_data['Display Name'] == 'F02', 'Display Name'] = 'E02'
    plate_data.loc[plate_data['Display Name'] == 'F02_temp', 'Display Name'] = 'F02'
    
    return plate_data

# Apply the well switch to Plate 3
plates3 = switch_wells(plates3)

# Generate separate heatmaps for each plate
generate_heatmap(plates, 'Plate 1')
generate_heatmap(plates2, 'Plate 2')
generate_heatmap(plates3, 'Plate 3')


coefficient of varience 

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Load the plate data
plates = pd.read_pickle('R:/toshiba usb/merged_p1_30k.pkl')
plates2 = pd.read_pickle('R:/toshiba usb/merged_p2_30k.pkl')
plates3 = pd.read_pickle('R:/toshiba usb/merged_p3_30k.pkl')

# Print original counts for D02
print("E05 counts:")
print("Plate 1:", plates[plates['Display Name'] == 'E05'].shape[0])
print("Plate 2:", plates2[plates2['Display Name'] == 'E05'].shape[0])
print("Plate 3:", plates3[plates3['Display Name'] == 'E05'].shape[0])

# Modify the count for well D02 in Plate 1 to 5% of its original count
d02_count_plate1 = plates[plates['Display Name'] == 'D02'].shape[0]
new_d02_count_plate1 = int(d02_count_plate1 * 0.05)

# Update Plate 1 to have 5% of the original count for D02
plates.loc[plates['Display Name'] == 'D02', 'Display Name'] = 'D02_temp'
plates = plates.append(pd.DataFrame({'Display Name': ['D02'] * new_d02_count_plate1}), ignore_index=True)
plates = plates[plates['Display Name'] != 'D02_temp']

# Print the updated counts for D02 in Plate 1
print("\nUpdated D02 counts:")
print("Plate 1:", plates[plates['Display Name'] == 'D02'].shape[0])

# Function to calculate the coefficient of variation across the plates
letter = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
num = ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12']

def calculate_cv_across_plates(plates_list):
    wells_data = []
    
    for plate in plates_list:
        plate_well_data = []
        for x in range(len(letter)):  # Changed loop order to match reshape
            for i in range(len(num)):
                well = str(letter[x]) + str(num[i])
                well_count = plate[plate['Display Name'] == well].shape[0]
                plate_well_data.append(well_count)
        wells_data.append(plate_well_data)

    wells_df = pd.DataFrame(wells_data, columns=[str(letter[x]) + str(num[i]) for x in range(len(letter)) for i in range(len(num))])
    
    # Calculate mean and standard deviation for each well
    mean_counts = wells_df.mean(axis=0)
    std_counts = wells_df.std(axis=0)

    # Calculate the coefficient of variation (CV)
    cv = std_counts / mean_counts
    return cv, mean_counts, std_counts

# Calculate the CV across the three plates
cv_values, mean_values, std_values = calculate_cv_across_plates([plates, plates2, plates3])

# Reshape the CV values into a matrix format
cv_matrix = cv_values.values.reshape(8, 12)

# Plot the heatmap for the CV values
plt.figure(figsize=(10, 6))  # Slightly larger figure size
sns.heatmap(
    cv_matrix,
    cmap="YlGnBu",  # YlGnBu color map
    annot=True,  # Show annotations in the cells
    fmt=".2f",  # Format annotations as integers
    cbar_kws={'label': 'Cell Counts'},  # Color bar label
    annot_kws={'size': 10},  # Increase annotation font size
    xticklabels=num,  # Column labels
    yticklabels=letter,  # Row labels
)
plt.title('Coefficient of Variation Across Wells', fontsize=14)  # Larger title font size
plt.xlabel('Column Number', fontsize=12)  # Larger x-axis label font
plt.ylabel('Row Letter', fontsize=12)  # Larger y-axis label font
plt.xticks(fontsize=10)  # Larger x-axis ticks font
plt.yticks(fontsize=10)  # Larger y-axis ticks font
plt.show()

# Print out the mean, standard deviation, and CV for each well
result_df = pd.DataFrame({
    'Mean': mean_values,
    'Standard Deviation': std_values,
    'CV': cv_values
}, index=[str(letter[x]) + str(num[i]) for x in range(len(letter)) for i in range(len(num))])

print("\nMean, Standard Deviation, and CV for each well:")
print(result_df)


visualisation of raw and log transformed data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Load the plate data
plates = pd.read_pickle('R:/toshiba usb/merged_p1_30k.pkl')
plates2 = pd.read_pickle('R:/toshiba usb/merged_p2_30k.pkl')
plates3 = pd.read_pickle('R:/toshiba usb/merged_p3_30k.pkl')

# Function to generate histograms and calculate Cohen's d for multiple wells on the same plot
def generate_combined_histograms_and_stats_with_cohen_d(plate_data, plate_name, wells, hypothesized_mean=0):
    plt.figure(figsize=(10, 6))
    stats_data = []

    # Loop through each well and add its histogram to the plot
    for well in wells:
        # Extract "Feature Area" for the current well
        well_data = plate_data[plate_data['Display Name'] == well]['Feature Area']

        # Check if there is data for the well
        if well_data.empty:
            print(f"No data available for well {well} in {plate_name}")
            continue

        # Plot histogram for the current well
        sns.histplot(well_data, bins=500, kde=True, label=f'Well {well}', alpha=0.6)

        # Log-transform the data
        well_data_log = np.log(well_data + 1)  # Add 1 to avoid log(0)

        # Calculate descriptive statistics
        if len(well_data_log) > 0:
            mean_log = np.mean(well_data_log)
            std_log = np.std(well_data_log)
            
            # Calculate Cohen's d
            cohen_d = (mean_log - hypothesized_mean) / std_log if std_log != 0 else np.nan
            
            # Add descriptive statistics to list
            stats_data.append([well, mean_log, std_log, cohen_d])
            
            # Annotate the plot with Cohen's d
            plt.text(mean_log, 10, f"Cohen's d = {cohen_d:.2f}", color='black', fontsize=12)

    # Add plot titles and labels
    plt.title(f'{plate_name} - Combined Histogram of Feature Area for Wells {", ".join(wells)}')
    plt.xlabel('Feature Area')
    plt.ylabel('Frequency')
    plt.xlim(-2000, 2000)
    plt.legend()
    plt.show()

    # Now for the log-transformed data
    plt.figure(figsize=(10, 6))

    for well in wells:
        # Extract "Feature Area" for the current well
        well_data = plate_data[plate_data['Display Name'] == well]['Feature Area']

        # Check if there is data for the well
        if well_data.empty:
            print(f"No data available for well {well} in {plate_name}")
            continue

        # Log-transform the "Feature Area" data
        well_data_log = np.log(well_data + 1)  # Add 1 to avoid log(0)

        # Plot histogram for the current well (log-transformed data)
        sns.histplot(well_data_log, bins=100, kde=True, label=f'Well {well} (Log)', alpha=0.6)

    # Add plot titles and labels for log-transformed data
    plt.title(f'{plate_name} - Combined Histogram of Log-Transformed Feature Area for Wells {", ".join(wells)}')
    plt.xlabel('Log(Feature Area + 1)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()

    # Create a DataFrame for the statistics and export it
    stats_df = pd.DataFrame(stats_data, columns=['Well', 'Mean (Log)', 'Std Dev (Log)', "Cohen's d"])
    stats_df.to_csv(f'{plate_name}_log_stats_with_cohen_d.csv', index=False)
    print(f'Descriptive statistics saved to {plate_name}_log_stats_with_cohen_d.csv')

# Define the wells to plot histograms for
wells_to_plot = ["B03", "C03", "D03", "E03", "F03", "G03"]

# Generate combined histograms, calculate Cohen's d, and descriptive statistics for the specified wells in each plate
generate_combined_histograms_and_stats_with_cohen_d(plates, 'Plate 1', wells_to_plot)
generate_combined_histograms_and_stats_with_cohen_d(plates2, 'Plate 2', wells_to_plot)
generate_combined_histograms_and_stats_with_cohen_d(plates3, 'Plate 3', wells_to_plot)

Z-score normalisation

**Use script labelled "Z-scoring"**

Prep-UMAP data

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Mon Oct 14 19:49:47 2024

@author: Engineer
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from umap import UMAP

# Load the full dataset
data = pd.read_pickle('R:/toshiba usb/plate_all_zs_new.pkl')

# Extract each plate separately
plates = ["plate_30k_1", "plate_30k_2", "plate_30k_3"]
plate_data = {plate: data[data["Plate_num"] == plate].copy() for plate in plates}

# Columns for PCA
Columns = np.array([
    'Region:Circularity',
    'Region:Clumpiness',
    'Region:Diameter, Mean',
    'Region:Fractal Dimension',
    'Region:Heterogeneity',
    'Region:Hole Area',
    'Region:Hole Area Ratio',
    'Region:Roundness',
    'Nuclear Area',
    'Percent Area Parent',
    'Feature Area',
    'Region:MCA Intensity of Feature (mean)(Sum|Child 2)',
    'Region:MCA Intensity of Feature (mean)(Sum|Child 3)'
])

# Process each plate
for plate_name, plate_df in plate_data.items():
    # Extract the data for PCA
    MAT = np.array(plate_df[Columns])
    MAT = np.nan_to_num(MAT)

    # Perform PCA with whitening
    pca = PCA(n_components=Columns.size, svd_solver='randomized', whiten=True)
    MAT = pca.fit_transform(MAT)

    # Perform UMAP for dimensionality reduction
    umap_2d = UMAP(n_components=2, n_jobs=-1, n_neighbors=50)
    proj_2d = umap_2d.fit_transform(MAT)

    # Add UMAP results back to the plate data
    plate_df["umap_1"] = proj_2d[:, 0]
    plate_df["umap_2"] = proj_2d[:, 1]

    # Save the processed plate data
    output_file = f"R:/toshiba usb/{plate_name}_umap.csv"
    plate_df.to_csv(output_file, index=False)
    print(f"Saved UMAP data for {plate_name} to {output_file}")

    # Plot the UMAP results for control wells (D03 and D07)
    Controls = ["D03", "D07"]
    for control in Controls:
        Fil = plate_df["Display Name"] == control
        plt.scatter(
            plate_df[Fil]["umap_1"],
            plate_df[Fil]["umap_2"],
            s=0.5,
            label=control
        )
    plt.legend()
    plt.title(f"UMAP Projection for {plate_name}")
    plt.show()


UMAP all particles one concentration 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap

# Load and concatenate the three plates
plate_1 = pd.read_csv(r"R:/toshiba usb/plate_30k_1_umap.csv")
plate_2 = pd.read_csv(r"R:/toshiba usb/plate_30k_2_umap.csv")
plate_3 = pd.read_csv(r"R:/toshiba usb/plate_30k_3_umap.csv")

# Concatenate the plates
data = pd.concat([plate_1, plate_2, plate_3], ignore_index=True)

# Check if UMAP columns exist; if not, compute them
if "umap_1" not in data.columns or "umap_2" not in data.columns:
    features = data.select_dtypes(include=[np.number])  # Use numerical columns
    umap_model = umap.UMAP(random_state=42)
    umap_results = umap_model.fit_transform(features)
    data["umap_1"] = umap_results[:, 0]
    data["umap_2"] = umap_results[:, 1]

# Particle mapping based on the last digit
particle_map = {
    "04": "CuO",
    "05": "CeO2",
    "06": "SiO2",
    "07": "ZnO"
}

# Plot settings
font_size = 14  # Adjust font size
marker_size = 1.5  # Adjust marker size
legend_marker_size = 6  # Size of dots in the legend

# Concentration mapping
concentration_map = {
    'B': '100µg/ml',
    'C': '33µg/ml',
    'D': '11µg/ml',
    'E': '3µg/ml',
    'F': '1µg/ml',
    'G': '0.4µg/ml'
}

# Control well
control_well = "D03"

# Create plots for each particle type
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex="all", sharey="all")  # Smaller figure size
axes = axes.flatten()

wells_to_plot = ["D04", "D05", "D06", "D07"]

# Iterate over each particle type and plot
for idx, (particle_key, particle) in enumerate(particle_map.items()):
    # Plot for each particle type
    ax = axes[idx]
    
    Fil = (data["Display Name"] == control_well)
    Control_x = data[Fil]["umap_1"]
    Control_y = data[Fil]["umap_2"]

    for well in wells_to_plot:
        if particle_key in well:
            Fil = (data["Display Name"] == well)
            concentration_label = concentration_map.get(well[0], "Unknown Concentration")
            ax.scatter(Control_x, Control_y, s=marker_size, label=f"Control", alpha=0.5)
            ax.scatter(data[Fil]["umap_1"], data[Fil]["umap_2"], s=marker_size, label=f" ({particle}) - {concentration_label}", alpha=0.8)

    ax.set_xlabel("UMAP 1", fontsize=font_size)
    ax.set_ylabel("UMAP 2", fontsize=font_size)
    ax.legend(fontsize=font_size, scatterpoints=1, markerscale=legend_marker_size)
    ax.tick_params(axis='both', labelsize=font_size)
    #ax.set_title(f"Particle: {particle}", fontsize=font_size + 2)

# Adjust layout to accommodate the titles
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


UMAP single particle analysis

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Mon Oct 14 19:37:09 2024

@author: Engineer
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import umap

# Load and concatenate the three plates
plate_1 = pd.read_csv(r"R:/toshiba usb/plate_30k_1_umap.csv")
plate_2 = pd.read_csv(r"R:/toshiba usb/plate_30k_2_umap.csv")
plate_3 = pd.read_csv(r"R:/toshiba usb/plate_30k_3_umap.csv")

# Concatenate the plates
data = pd.concat([plate_1, plate_2, plate_3], ignore_index=True)

# Check if UMAP columns exist; if not, compute them
if "umap_1" not in data.columns or "umap_2" not in data.columns:
    features = data.select_dtypes(include=[np.number])  # Use numerical columns
    umap_model = umap.UMAP(random_state=42)
    umap_results = umap_model.fit_transform(features)
    data["umap_1"] = umap_results[:, 0]
    data["umap_2"] = umap_results[:, 1]

# Particle mapping based on the last digit
particle_map = {
    "04": "CuO",
    "05": "CeO2",
    "06": "SiO2",
    "07": "ZnO"
}

# Plot settings
font_size = 14  # Adjust font size
marker_size = 1.5  # Adjust marker size
legend_marker_size = 6  # Size of dots in the legend

# Concentration mapping
concentration_map = {
    'B': '100µg/ml',
    'C': '33µg/ml',
    'D': '11µg/ml',
    'E': '3µg/ml',
    'F': '1µg/ml',
    'G': '0.4µg/ml'
}

# Control well
control_well = "D03"

# First set of plots
particle = particle_map.get("04", "Unknown")  # Get particle type based on "04"
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex="all", sharey="all")  # Smaller figure size
axes = axes.flatten()

Fil = (data["Display Name"] == control_well)
Control_x = data[Fil]["umap_1"]
Control_y = data[Fil]["umap_2"]

wells_to_plot = ["D04", "D05", "D06", "D07"]

for i, ax in enumerate(axes):
    if i >= len(wells_to_plot):
        break
    well = wells_to_plot[i]
    Fil = (data["Display Name"] == well)
    ax.scatter(Control_x, Control_y, s=marker_size, label=f"{control_well} (Control)")
    ax.scatter(data[Fil]["umap_1"], data[Fil]["umap_2"], s=marker_size, label=f"{well}")
    ax.set_xlabel("UMAP 1", fontsize=font_size)
    ax.set_ylabel("UMAP 2", fontsize=font_size)
    ax.legend(fontsize=font_size, scatterpoints=1, markerscale=legend_marker_size)
    ax.tick_params(axis='both', labelsize=font_size)

# Add a title for the first set of plots
fig.suptitle(f"Particle: {particle}", fontsize=font_size + 2)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to accommodate the title
plt.show()

# Second set of plots
particle = particle_map.get("04", "Unknown")  # Get particle type based on "07"
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex="all", sharey="all")  # Slightly smaller figure size
axes = axes.flatten()

Fils = ["B07", "C07", "D07", "E07", "F07", "G07"]

for i, ax in enumerate(axes):
    if i >= len(Fils):
        break
    well = Fils[i]
    well_letter = well[0]  # Extract the first letter to map concentration
    concentration = concentration_map.get(well_letter, "Unknown")
    Fil = (data["Display Name"] == well)
    ax.scatter(Control_x, Control_y, s=marker_size, label=f"{control_well} (Control)")
    ax.scatter(data[Fil]["umap_1"], data[Fil]["umap_2"], s=marker_size, label=f"{concentration}")
    ax.set_xlabel("UMAP 1", fontsize=font_size)
    ax.set_ylabel("UMAP 2", fontsize=font_size)
    ax.legend(fontsize=font_size, scatterpoints=1, markerscale=legend_marker_size)
    ax.tick_params(axis='both', labelsize=font_size)

# Add a title for the second set of plots
fig.suptitle(f"{particle}", fontsize=font_size + 2)

plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to accommodate the title
plt.show()


PCA all particles 

In [ ]:
# -*- coding: utf-8 -*-
"""
Created on Fri Oct 18 15:49:20 2024

@author: Engineer
"""

import pickle
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# Load data from the specified path
plates = pd.read_pickle('R:/toshiba usb/plate_all_zs_new.pkl')

# List of plates to be analyzed
p8n = ['plate_30k_1', 'plate_30k_2', 'plate_30k_3']

# Filter and reset index
plates = plates.set_index(['Plate_num'])
plates_1 = plates.loc[plates.index.isin(p8n)]
plates_1 = plates_1.reset_index()

# Set index to 'Display Name'
plates_2 = plates_1.set_index(['Display Name'])

# Filter wells with "D" in their names (11ug/ml concentration)
plates_2 = plates_2[plates_2.index.str.startswith('D')]

# Drop unnecessary columns with error handling
columns_to_drop = [
    'Plate_num', 'Display Name', 'Region:Clumpiness', 
    'Region:Fractal Dimension', 'Concentration', 
    'Region:Heterogeneity', 'Region:Roundness'
]

# Reset index to ensure all columns are available for dropping
plates_2 = plates_2.reset_index()

# Drop columns (ignore missing ones to prevent errors)
numeric_columns = plates_2.drop(labels=columns_to_drop, axis=1, errors='ignore')

# Impute missing values with the median
imputer = SimpleImputer(strategy='median')
numeric_columns = pd.DataFrame(
    imputer.fit_transform(numeric_columns),
    columns=numeric_columns.columns
)

# Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_columns)

# Perform PCA
pca = PCA(n_components=5)
principal_components = pca.fit_transform(scaled_data)
principal_df = pd.DataFrame(data=principal_components, columns=[f'PC {i+1}' for i in range(5)])

# Add nanoparticle types as categories
plates_2 = plates_2.reset_index()
plates_2.loc[plates_2['Display Name'].str.contains('03'), 'Display Name'] = 'Untreated_3'
plates_2.loc[plates_2['Display Name'].str.contains('04'), 'Display Name'] = 'CuO'
plates_2.loc[plates_2['Display Name'].str.contains('05'), 'Display Name'] = 'Ce(iv)O'
plates_2.loc[plates_2['Display Name'].str.contains('06'), 'Display Name'] = 'SiO2'
plates_2.loc[plates_2['Display Name'].str.contains('07'), 'Display Name'] = 'ZnO'

# Combine PCA results with the nanoparticle labels
final_df = pd.concat([principal_df, plates_2[['Display Name']]], axis=1)

# Filter samples of interest
samples = ['Untreated_3', 'CuO', 'Ce(iv)O', 'SiO2', 'ZnO']
final_df = final_df[final_df['Display Name'].isin(samples)]

# Plot the pairplot using seaborn
sns.set_theme(style="ticks")
sns.set_context("paper", rc={"axes.labelsize": 24})
f = sns.pairplot(final_df, hue="Display Name", plot_kws={'alpha': 0.7, 's': 40})

# Adjust font size for axis ticks and legend
plt.tick_params(axis='both', labelsize=16)
plt.legend(fontsize=16)

# Show the plot
plt.show()

# Plot loadings (feature contributions to each principal component)
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=[f'PC {i+1}' for i in range(pca.n_components_)],
    index=numeric_columns.columns
)
plt.figure(figsize=(10, 8))
sns.heatmap(loadings, cmap='coolwarm', annot=True, fmt='.2f', annot_kws={'size': 14})

# Adjust font size for ticks and labels
plt.tick_params(axis='both', labelsize=16)
plt.xlabel('Principal Components', fontsize=18)
plt.ylabel('Features', fontsize=18)
plt.title('Loadings for Principal Components', fontsize=20)

plt.show()

# Explained variance and cumulative variance plot
explained_variance = pca.explained_variance_ratio_ * 100
cumulative_variance = np.cumsum(explained_variance)

# Create a DataFrame for explained variance
explained_variance_df = pd.DataFrame({
    'Principal Component': [f'PC {i+1}' for i in range(len(explained_variance))],
    'Individual Variance': explained_variance,
    'Cumulative Variance': cumulative_variance
})

# Plot explained variance
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Bar plot for individual explained variance
ax = sns.barplot(x='Principal Component', y='Individual Variance', data=explained_variance_df, color='skyblue', alpha=0.6)

# Line plot for cumulative explained variance
sns.lineplot(
    x='Principal Component', y='Cumulative Variance', 
    data=explained_variance_df, marker='o', color='orange', 
    label='Cumulative Explained Variance', ax=ax
)

# Titles and labels
plt.title(f'Cumulative Explained Variance: {cumulative_variance[-1]:.2f}%', fontsize=16)
plt.ylabel('Variance Explained (%)', fontsize=18)
plt.xlabel('Principal Component', fontsize=18)
plt.legend(loc='upper left', fontsize=16)

# Adjust y-axis limits
ax.set_ylim(0, 110)

# Show the plot
plt.show()


PCA single particles all concentrations 

In [ ]:
from os import chdir
import sys
import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# Change directory and append paths
sys.path.append('../')
chdir('../')

# Load data
plates = pd.read_pickle('R:/toshiba usb/plate_all_zs_new.pkl')

# Define plates and mapping
p8n = ['plate_30k_1', 'plate_30k_2', 'plate_30k_3']
plates = plates.set_index(['Plate_num'])
plates_1 = plates.loc[plates.index.isin(p8n)].reset_index()
plates_2 = plates_1.set_index(['Display Name'])

# Well and particle mapping
num = ['03', '04', '05', '06', '07']
particle_map = {'03': 'Control (C03)', '04': 'CuO', '05': 'CeO2', '06': 'SiO2', '07': 'ZnO'}
concentration_map = {'B': '100µg/ml', 'C': '33µg/ml', 'D': '11µg/ml', 'E': '3µg/ml', 'F': '1µg/ml', 'G': '0.4µg/ml'}

# Loop through particles
for i in tqdm(range(len(num)), desc="Processing wells", ncols=100):
    well = f'B{num[i]}'
    particle_name = particle_map[num[i]]

    # Select wells for the current nanoparticle
    plates_3 = plates_2.loc[plates_2.index.str.contains(f'B{num[i]}|C{num[i]}|D{num[i]}|E{num[i]}|F{num[i]}|G{num[i]}')].reset_index()

    # Label concentrations
    plates_3.loc[plates_3['Display Name'].str.contains('B'), 'Concentration'] = '100µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('C'), 'Concentration'] = '33µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('D'), 'Concentration'] = '11µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('E'), 'Concentration'] = '3µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('F'), 'Concentration'] = '1µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('G'), 'Concentration'] = '0.4µg/ml'
    plates_3 = plates_3.dropna()

    app_data_inner = []
    target_values = []

    # Process each plate
    for n in range(len(p8n)):
        c03_data = plates_2.loc[(plates_2.index.str.contains('C03')) & (plates_2['Plate_num'] == p8n[n])].reset_index()
        plates_4 = plates_3.set_index(['Plate_num'])
        plates_4_n = plates_4.loc[plates_4.index.isin([p8n[n]])].reset_index()

        plates_combined = pd.concat([plates_4_n, c03_data], ignore_index=True)
        labels = list(plates_4_n['Concentration']) + ['Control'] * len(c03_data)
        target_values.extend(labels)

        numeric_columns = plates_combined.drop(labels=['Plate_num', 'Display Name', 'Region:Clumpiness', 'Region:Fractal Dimension', 'Concentration', 'Region:Heterogeneity', 'Region:Roundness'], axis=1)
        scaled_data = StandardScaler().fit_transform(numeric_columns)
        app_data_inner.append(scaled_data)

    combined_data = np.concatenate(app_data_inner, axis=0)
    pca = PCA(n_components=5)
    principal_components = pca.fit_transform(combined_data)
    principal_df = pd.DataFrame(data=principal_components, columns=['PC 1', 'PC 2', 'PC 3', 'PC4', 'PC5'])
    principal_df['Concentration'] = target_values

    # Set Seaborn theme and context
    sns.set_theme(style="ticks")
    sns.set_context("paper", rc={"axes.labelsize": 18})

    # Plot the pairplot
    g = sns.pairplot(
        principal_df, hue='Concentration', height=3,
        x_vars=['PC 1', 'PC 2', 'PC 3', 'PC4', 'PC5'], y_vars=['PC 1', 'PC 2', 'PC 3', 'PC4', 'PC5'], plot_kws={'alpha': 0.7}
    )
    g.fig.suptitle(f'{particle_name}: PCA of All Concentrations', fontsize=16)
    g.tight_layout()
    g.fig.subplots_adjust(top=0.95)

    # Adjust font size for ticks and legend
    for ax in g.axes.flatten():
        ax.tick_params(axis='both', labelsize=14)  # Increase font size for axis ticks
    
    plt.legend(fontsize=14)  # Increase font size for legend
    plt.show()


PCA's cumulative varience and loadings

In [ ]:
from os import chdir
import sys
import pickle
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

# Change directory and append paths
sys.path.append('../')
chdir('../')

# Load data
plates = pd.read_pickle('R:/toshiba usb/plate_all_zs_new.pkl')

# Define plates and mapping
p8n = ['plate_30k_1', 'plate_30k_2', 'plate_30k_3']
plates = plates.set_index(['Plate_num'])
plates_1 = plates.loc[plates.index.isin(p8n)].reset_index()
plates_2 = plates_1.set_index(['Display Name'])

# Well and particle mapping
num = ['03', '04', '05', '06', '07']
particle_map = {'03': 'Control (C03)', '04': 'CuO', '05': 'CeO2', '06': 'SiO2', '07': 'ZnO'}
concentration_map = {'B': '100µg/ml', 'C': '33µg/ml', 'D': '11µg/ml', 'E': '3µg/ml', 'F': '1µg/ml', 'G': '0.4µg/ml'}

# Create a DataFrame to store cumulative variance for all particles and concentrations
cumulative_variance_df = pd.DataFrame(columns=['Particle', 'Concentration', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'Cumulative Variance'])

# Loop through each nanoparticle
for i in tqdm(range(len(num)), desc="Processing wells", ncols=100):
    well = f'B{num[i]}'
    particle_name = particle_map[num[i]]  # Get the corresponding particle name

    # Select wells for the current nanoparticle
    plates_3 = plates_2.loc[plates_2.index.str.contains(f'B{num[i]}|C{num[i]}|D{num[i]}|E{num[i]}|F{num[i]}|G{num[i]}')].reset_index()

    # Label concentrations based on the well rows (B-G)
    plates_3.loc[plates_3['Display Name'].str.contains('B'), 'Concentration'] = '100µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('C'), 'Concentration'] = '33µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('D'), 'Concentration'] = '11µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('E'), 'Concentration'] = '3µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('F'), 'Concentration'] = '1µg/ml'
    plates_3.loc[plates_3['Display Name'].str.contains('G'), 'Concentration'] = '0.4µg/ml'

    # Drop missing data
    plates_3 = plates_3.dropna()

    app_data_inner = []
    target_values = []  # Reset target values for each well

    # Loop through each plate
    for n in range(len(p8n)):
        # Get control data (C03) specific to the current plate (p8n[n])
        c03_data = plates_2.loc[(plates_2.index.str.contains('C03')) & (plates_2['Plate_num'] == p8n[n])].reset_index()
        
        plates_4 = plates_3.set_index(['Plate_num'])
        plates_4_n = plates_4.loc[plates_4.index.isin([p8n[n]])].reset_index()

        # Concatenate C03 data before dropping unnecessary columns and scaling
        plates_combined = pd.concat([plates_4_n, c03_data], ignore_index=True)

        # Append the concentration labels to target_values for later labelling in plots
        labels = list(plates_4_n['Concentration']) + ['Control'] * len(c03_data)
        target_values.extend(labels)

        # Now drop unwanted columns and scale the numeric data
        numeric_columns = plates_combined.drop(labels=['Plate_num', 'Display Name', 'Region:Clumpiness', 'Region:Fractal Dimension', 'Concentration'], axis=1)
        scaled_data = StandardScaler().fit_transform(numeric_columns)

        # Store scaled data
        app_data_inner.append(scaled_data)

    # Concatenate all data from app_data_inner
    combined_data = np.concatenate(app_data_inner, axis=0)

    # Perform PCA on the combined data (control and iterated wells together)
    pca = PCA(n_components=5)
    principal_components = pca.fit_transform(combined_data)

    # Convert the principal components to a DataFrame
    principal_df = pd.DataFrame(data=principal_components, columns=['PC 1', 'PC 2', 'PC 3', 'PC 4', 'PC 5'])

    # Add the concentration labels to the principal components DataFrame
    principal_df['Concentration'] = target_values

    ### Explained Variance and Cumulative Variance Calculation ###
    explained_variance = pca.explained_variance_ratio_
    csum = np.cumsum(explained_variance)

    # Append the explained variance and cumulative variance for each principal component
    for i, (ind_var, cum_var) in enumerate(zip(explained_variance, csum)):
        for conc in set(target_values):
            new_row = {
                'Particle': particle_name,
                'Concentration': conc,
                'Principal Component': f'PC {i+1}',
                'Individual Variance': ind_var,
                'Cumulative Variance': cum_var
            }
            cumulative_variance_df = pd.concat([cumulative_variance_df, pd.DataFrame([new_row])], ignore_index=True)

    ### Loading plot (feature contributions to each principal component) ###
    loadings = pd.DataFrame(pca.components_.T, columns=[f'PC {i+1}' for i in range(pca.n_components_)], index=numeric_columns.columns)

    # Plot loading heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(loadings, cmap='coolwarm', annot=True, fmt='.2f',
                xticklabels=True, yticklabels=True,
                annot_kws={'size': 14})  # Adjust font size for the values
    plt.title(f'Loading Plot for {particle_name}', fontsize=16)  # Title size
    plt.xticks(fontsize=14)  # X-axis tick size
    plt.yticks(fontsize=14)  # Y-axis tick size
    plt.show()

    ### Explained Variance and Cumulative Variance Plot ###
    exp1 = explained_variance * 100  # Individual explained variance ratio
    csum_per = csum * 100  # Cumulative variance in percentage

    # Create a DataFrame for Seaborn
    explained_variance_df = pd.DataFrame({
        'Principal Component': [f'PC {i+1}' for i in range(len(exp1))],
        'Individual Variance': exp1,
        'Cumulative Variance': csum_per
    })

    # Set the Seaborn theme for consistency in style
    sns.set_theme(style="whitegrid")

    # Create a figure for the explained variance plot
    plt.figure(figsize=(10, 6))

    # Bar plot for individual explained variance
    ax = sns.barplot(x='Principal Component', y='Individual Variance', data=explained_variance_df, color='skyblue', alpha=0.6)

    # Line plot for cumulative explained variance
    sns.lineplot(x='Principal Component', y='Cumulative Variance', data=explained_variance_df, marker='o', color='orange', label='Cumulative Explained Variance', ax=ax)

    # Title and labels with larger font sizes
    plt.title(f'{particle_name} - Cumulative Explained Variance: {csum_per[4]:.2f}%', fontsize=16)
    plt.ylabel('Variance Explained (%)', fontsize=14)
    plt.xlabel('Principal Component', fontsize=14)

    # Adjust tick font sizes
    ax.tick_params(axis='x', labelsize=14)
    ax.tick_params(axis='y', labelsize=14)

    # Adjust legend font size
    plt.legend(loc='upper left', fontsize=12)

    # Adjust y-axis to ensure both the bar and line are visible
    ax.set_ylim(0, 110)  # Set y-axis to go beyond 100% for better visibility

    plt.show()

# Save the cumulative variance DataFrame for later use
cumulative_variance_df.to_csv('R:/toshiba usb/cumulative_variance_results.csv', index=False)


**MACHINE LEARNING APPROACH**

XGBOOST

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': 'CeO₂',
    '06': 'SiO₂',
    '07': 'ZnO'
}

# Dosage dictionary
dosage_dict = {
    'B': '100\u03bcg/ml',
    'C': '33\u03bcg/ml',
    'D': '11\u03bcg/ml',
    'E': '3\u03bcg/ml',
    'F': '1\u03bcg/ml',
    'G': '0.4\u03bcg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Function to train and evaluate binary classifier
def train_evaluate_classifier(compound_data, control_data):
    # Dynamically decide whether to undersample or oversample
    if len(control_data) > len(compound_data):
        # Undersample the control data to match the compound data size
        control_data = resample(control_data, replace=False, n_samples=len(compound_data), random_state=42)
    elif len(control_data) < len(compound_data):
        # Oversample the control data to match the compound data size
        control_data = resample(control_data, replace=True, n_samples=len(compound_data), random_state=42)

    # Prepare the data
    X_compound = compound_data[features]
    y_compound = pd.Series(1, index=X_compound.index)
    X_control = control_data[features]
    y_control = pd.Series(0, index=X_control.index)

    X = pd.concat([X_compound, X_control])
    y = pd.concat([y_compound, y_control])

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train XGBoost classifier
    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    xgb_clf.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = xgb_clf.predict(X_test_scaled)
    y_pred_proba = xgb_clf.predict_proba(X_test_scaled)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return accuracy, precision, recall, f1, roc_auc, len(compound_data), len(control_data)

# Iterate through all dosages
all_results = {}
for dosage_code, dosage_label in dosage_dict.items():
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Identify control samples
    control_samples = dosage_data[dosage_data['Compound'] == '03']
    
    # Initialize results for this dosage
    results = {}
    
    # Train and evaluate classifiers for each compound at the current dosage
    for compound in dosage_data['Compound'].unique():
        if compound in compound_dict:  # Only process compounds in our dictionary
            compound_data = dosage_data[dosage_data['Compound'] == compound]
            if not compound_data.empty:
                accuracy, precision, recall, f1, roc_auc, n_compound, n_control = train_evaluate_classifier(compound_data, control_samples)
                results[compound] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'roc_auc': roc_auc,
                    'n_compound': n_compound,
                    'n_control': n_control
                }
    
    all_results[dosage_label] = results

    # Plot results for the current dosage
    compounds = list(results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    print("\nSample sizes per compound at dosage level:", dosage_label)
    print(f"{'Compound':<15}{'Compound Samples':<20}{'Control Samples':<20}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['n_compound']:<20}{metrics_values['n_control']:<20}")

   # Adjust figure size and font sizes
    fig, ax1 = plt.subplots(figsize=(10, 6))  # Reduced figure size

    # Plot metrics
    for metric in metrics:
        values = [results[compound][metric] for compound in compounds]
        ax1.plot(compounds, values, marker='o', label=metric)

    # Axis settings
    ax1.set_xlabel('Compound', fontsize=16)  # Increased font size
    ax1.set_ylabel('Metric Value', fontsize=16)  # Increased font size
    ax1.tick_params(axis='both', which='major', labelsize=14)  # Increased tick label size
    ax1.set_ylim(0, 1)
    ax1.grid(True)

    # Create a second y-axis for sample sizes
    ax2 = ax1.twinx()

    # Plot sample sizes as bars
    x = np.arange(len(compounds))
    width = 0.35
    control_samples = [results[compound]['n_control'] for compound in compounds]
    compound_samples = [results[compound]['n_compound'] for compound in compounds]

    ax2.bar(x - width/2, control_samples, width, label='Control Samples', alpha=0.5, color='lightblue')
    ax2.bar(x + width/2, compound_samples, width, label='Compound Samples', alpha=0.5, color='lightgreen')

    # Axis settings for second y-axis
    ax2.set_ylabel('Number of samples', fontsize=14)  # Increased font size
    ax2.tick_params(axis='y', which='major', labelsize=12)  # Increased tick label size

    # Set x-axis ticks and labels
    plt.xticks(x, [compound_dict[compound] for compound in compounds], fontsize=12)  # Adjusted font size for x-ticks

    # Title and layout adjustments
    plt.title(f'Evaluation Metrics for Control (03) vs Compounds at {dosage_label}', fontsize=16)  # Increased title font size
    plt.tight_layout()

    # Move legends outside of the graph
    ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1), fontsize=12)  # For metrics
    ax2.legend(loc='upper left', bbox_to_anchor=(1.1, 0.6), fontsize=12)  # For sample sizes

    # Show the plot
    plt.show()

    # Print metrics summary
    print("\nMetrics summary:")
    print(f"{'Compound':<15}{'Accuracy':<15}{'Precision':<15}{'Recall':<15}{'F1-Score':<15}{'ROC AUC':<15}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['accuracy']:<15.4f}{metrics_values['precision']:<15.4f}{metrics_values['recall']:<15.4f}{metrics_values['f1']:<15.4f}{metrics_values['roc_auc']:<15.4f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': r'CeO$_2$',  # Use LaTeX for subscript
    '06': r'SiO$_2$',  # Use LaTeX for subscript
    '07': 'ZnO'
}



# Dosage dictionary
dosage_dict = {
    'B': '100μg/ml',
    'C': '33μg/ml',
    'D': '11μg/ml',
    'E': '3μg/ml',
    'F': '1μg/ml',
    'G': '0.4μg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Create figure for subplots
n_dosages = len(dosage_dict)
fig = plt.figure(figsize=(14, 4*n_dosages))  # Adjusted width to shorten X-axis

# Increase font scale for seaborn heatmap
sns.set(font_scale=1.2)  # Increase overall font size

# Iterate through all dosages
for i, (dosage_code, dosage_label) in enumerate(dosage_dict.items()):
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Prepare data for all compounds
    all_X = []
    all_y = []
    compound_sizes = {}
    
    # First, collect all data and determine sizes
    for compound_code in compound_dict.keys():
        compound_data = dosage_data[dosage_data['Compound'] == compound_code]
        if not compound_data.empty:
            compound_sizes[compound_code] = len(compound_data)
    
    # Find minimum size for balanced sampling
    if compound_sizes:
        min_size = min(compound_sizes.values())
        
        # Resample data for each compound
        for compound_code in compound_sizes.keys():
            compound_data = dosage_data[dosage_data['Compound'] == compound_code]
            if len(compound_data) > min_size:
                compound_data = resample(compound_data, replace=False, n_samples=min_size, random_state=42)
            elif len(compound_data) < min_size:
                compound_data = resample(compound_data, replace=True, n_samples=min_size, random_state=42)
            
            all_X.append(compound_data[features])
            all_y.extend([compound_dict[compound_code]] * len(compound_data))
    
    # Combine all data
    X = pd.concat(all_X)
    y = pd.Series(all_y)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Label encode the target variable
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train XGBoost classifier
    xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
    xgb_clf.fit(X_train_scaled, y_train_encoded)
    
    # Make predictions
    y_pred_encoded = xgb_clf.predict(X_test_scaled)
    
    # Convert predictions back to original labels
    y_pred = le.inverse_transform(y_pred_encoded)
    y_test = le.inverse_transform(y_test_encoded)
    
    # Create subplots for current dosage
    plt.subplot(n_dosages, 2, 2*i + 1)  # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(compound_dict.values()),
                yticklabels=list(compound_dict.values()),
                cbar=False)  # Disable color bar to reduce clutter
    plt.title(f'Confusion Matrix - {dosage_label}', fontsize=14)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    
    plt.subplot(n_dosages, 2, 2*i + 2)  # Feature Importance
    sorted_idx = np.argsort(xgb_clf.feature_importances_)
    plt.barh(np.array(features)[sorted_idx], xgb_clf.feature_importances_[sorted_idx], color='teal')
    plt.title(f'Feature Importance - {dosage_label}', fontsize=14)
    plt.xlabel('Importance Score', fontsize=12)
    plt.tick_params(axis='y', labelsize=10)  # Adjust y-axis tick font size for clarity
    
    # Print classification report
    print(f"\nClassification Report for {dosage_label}:")
    print(classification_report(y_test, y_pred))

plt.tight_layout()
plt.show()


KNN (K-nearest neighbours)

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': 'CeO₂',
    '06': 'SiO₂',
    '07': 'ZnO'
}

# Dosage dictionary
dosage_dict = {
    'B': '100\u03bcg/ml',
    'C': '33\u03bcg/ml',
    'D': '11\u03bcg/ml',
    'E': '3\u03bcg/ml',
    'F': '1\u03bcg/ml',
    'G': '0.4\u03bcg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Function to train and evaluate binary classifier
def train_evaluate_classifier(compound_data, control_data):
    # Dynamically decide whether to undersample or oversample
    if len(control_data) > len(compound_data):
        # Undersample the control data to match the compound data size
        control_data = resample(control_data, replace=False, n_samples=len(compound_data), random_state=42)
    elif len(control_data) < len(compound_data):
        # Oversample the control data to match the compound data size
        control_data = resample(control_data, replace=True, n_samples=len(compound_data), random_state=42)

    # Prepare the data
    X_compound = compound_data[features]
    y_compound = pd.Series(1, index=X_compound.index)
    X_control = control_data[features]
    y_control = pd.Series(0, index=X_control.index)

    X = pd.concat([X_compound, X_control])
    y = pd.concat([y_compound, y_control])

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # Train k-Nearest Neighbors classifier for binary classification
    k_neighbors = 5  # You can experiment with different values of k
    xgb_clf = KNeighborsClassifier(n_neighbors=k_neighbors)
    xgb_clf.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = xgb_clf.predict(X_test_scaled)
    y_pred_proba = xgb_clf.predict_proba(X_test_scaled)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return accuracy, precision, recall, f1, roc_auc, len(compound_data), len(control_data)

# Iterate through all dosages
all_results = {}
for dosage_code, dosage_label in dosage_dict.items():
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Identify control samples
    control_samples = dosage_data[dosage_data['Compound'] == '03']
    
    # Initialize results for this dosage
    results = {}
    
    # Train and evaluate classifiers for each compound at the current dosage
    for compound in dosage_data['Compound'].unique():
        if compound in compound_dict:  # Only process compounds in our dictionary
            compound_data = dosage_data[dosage_data['Compound'] == compound]
            if not compound_data.empty:
                accuracy, precision, recall, f1, roc_auc, n_compound, n_control = train_evaluate_classifier(compound_data, control_samples)
                results[compound] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'roc_auc': roc_auc,
                    'n_compound': n_compound,
                    'n_control': n_control
                }
    
    all_results[dosage_label] = results

    # Plot results for the current dosage
    compounds = list(results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    print("\nSample sizes per compound at dosage level:", dosage_label)
    print(f"{'Compound':<15}{'Compound Samples':<20}{'Control Samples':<20}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['n_compound']:<20}{metrics_values['n_control']:<20}")

   # Adjust figure size and font sizes
    fig, ax1 = plt.subplots(figsize=(10, 6))  # Reduced figure size

    # Plot metrics
    for metric in metrics:
        values = [results[compound][metric] for compound in compounds]
        ax1.plot(compounds, values, marker='o', label=metric)

    # Axis settings
    ax1.set_xlabel('Compound', fontsize=16)  # Increased font size
    ax1.set_ylabel('Metric Value', fontsize=16)  # Increased font size
    ax1.tick_params(axis='both', which='major', labelsize=14)  # Increased tick label size
    ax1.set_ylim(0, 1)
    ax1.grid(True)

    # Create a second y-axis for sample sizes
    ax2 = ax1.twinx()

    # Plot sample sizes as bars
    x = np.arange(len(compounds))
    width = 0.35
    control_samples = [results[compound]['n_control'] for compound in compounds]
    compound_samples = [results[compound]['n_compound'] for compound in compounds]

    ax2.bar(x - width/2, control_samples, width, label='Control Samples', alpha=0.5, color='lightblue')
    ax2.bar(x + width/2, compound_samples, width, label='Compound Samples', alpha=0.5, color='lightgreen')

    # Axis settings for second y-axis
    ax2.set_ylabel('Number of samples', fontsize=14)  # Increased font size
    ax2.tick_params(axis='y', which='major', labelsize=12)  # Increased tick label size

    # Set x-axis ticks and labels
    plt.xticks(x, [compound_dict[compound] for compound in compounds], fontsize=12)  # Adjusted font size for x-ticks

    # Title and layout adjustments
    plt.title(f'Evaluation Metrics for Control (03) vs Compounds at {dosage_label}', fontsize=16)  # Increased title font size
    plt.tight_layout()

    # Move legends outside of the graph
    ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1), fontsize=12)  # For metrics
    ax2.legend(loc='upper left', bbox_to_anchor=(1.1, 0.6), fontsize=12)  # For sample sizes

    # Show the plot
    plt.show()

    # Print metrics summary
    print("\nMetrics summary:")
    print(f"{'Compound':<15}{'Accuracy':<15}{'Precision':<15}{'Recall':<15}{'F1-Score':<15}{'ROC AUC':<15}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['accuracy']:<15.4f}{metrics_values['precision']:<15.4f}{metrics_values['recall']:<15.4f}{metrics_values['f1']:<15.4f}{metrics_values['roc_auc']:<15.4f}")


In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': r'CeO$_2$',  # Use LaTeX for subscript
    '06': r'SiO$_2$',  # Use LaTeX for subscript
    '07': 'ZnO'
}



# Dosage dictionary
dosage_dict = {
    'B': '100μg/ml',
    'C': '33μg/ml',
    'D': '11μg/ml',
    'E': '3μg/ml',
    'F': '1μg/ml',
    'G': '0.4μg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Create figure for subplots
n_dosages = len(dosage_dict)
fig = plt.figure(figsize=(14, 4*n_dosages))  # Adjusted width to shorten X-axis

# Increase font scale for seaborn heatmap
sns.set(font_scale=1.2)  # Increase overall font size
# Iterate through all dosages
for i, (dosage_code, dosage_label) in enumerate(dosage_dict.items()):
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Prepare data for all compounds
    all_X = []
    all_y = []
    compound_sizes = {}
    
    # First, collect all data and determine sizes
    for compound_code in compound_dict.keys():
        compound_data = dosage_data[dosage_data['Compound'] == compound_code]
        if not compound_data.empty:
            compound_sizes[compound_code] = len(compound_data)
    
    # Find minimum size for balanced sampling
    if compound_sizes:
        min_size = min(compound_sizes.values())
        
        # Resample data for each compound
        for compound_code in compound_sizes.keys():
            compound_data = dosage_data[dosage_data['Compound'] == compound_code]
            if len(compound_data) > min_size:
                compound_data = resample(compound_data, replace=False, n_samples=min_size, random_state=42)
            elif len(compound_data) < min_size:
                compound_data = resample(compound_data, replace=True, n_samples=min_size, random_state=42)
            
            all_X.append(compound_data[features])
            all_y.extend([compound_dict[compound_code]] * len(compound_data))
    
    # Combine all data
    X = pd.concat(all_X)
    y = pd.Series(all_y)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Label encode the target variable
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train k-Nearest Neighbors classifier
    k_neighbors = 5
    knn_clf = KNeighborsClassifier(n_neighbors=k_neighbors)
    knn_clf.fit(X_train_scaled, y_train_encoded)
    
    # Make predictions
    y_pred_encoded = knn_clf.predict(X_test_scaled)
    
    # Convert predictions back to original labels
    y_pred = le.inverse_transform(y_pred_encoded)
    y_test = le.inverse_transform(y_test_encoded)
    
    # Calculate permutation importance
    result = permutation_importance(
        knn_clf, X_test_scaled, y_test_encoded, 
        n_repeats=10, 
        random_state=42
    )
    
    feature_importance = result.importances_mean
    
    # Create subplots for current dosage
    plt.subplot(n_dosages, 2, 2*i + 1)  # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(compound_dict.values()),
                yticklabels=list(compound_dict.values()))
    plt.title(f'Confusion Matrix - {dosage_label}', fontsize=14)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    
    
    plt.subplot(n_dosages, 2, 2*i + 2)  # Feature Importance
    sorted_idx = np.argsort(feature_importance)
    plt.barh(np.array(features)[sorted_idx], feature_importance[sorted_idx], color='teal')
    plt.title(f'Feature Importance - {dosage_label}', fontsize=14)
    plt.xlabel('Importance Score', fontsize=12)
    plt.tick_params(axis='y', labelsize=10)  # Adjust y-axis tick font size for clarity
    
    # Print classification report
    print(f"\nClassification Report for {dosage_label}:")
    print(classification_report(y_test, y_pred))

plt.tight_layout()
plt.show()

MLP 

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': 'CeO₂',
    '06': 'SiO₂',
    '07': 'ZnO'
}

# Dosage dictionary
dosage_dict = {
    'B': '100μg/ml',
    'C': '33μg/ml',
    'D': '11μg/ml',
    'E': '3μg/ml',
    'F': '1μg/ml',
    'G': '0.4μg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Function to train and evaluate binary classifier
def train_evaluate_classifier(compound_data, control_data):
    # Dynamically decide whether to undersample or oversample
    if len(control_data) > len(compound_data):
        # Undersample the control data to match the compound data size
        control_data = resample(control_data, replace=False, n_samples=len(compound_data), random_state=42)
    elif len(control_data) < len(compound_data):
        # Oversample the control data to match the compound data size
        control_data = resample(control_data, replace=True, n_samples=len(compound_data), random_state=42)

    # Prepare the data
    X_compound = compound_data[features]
    y_compound = pd.Series(1, index=X_compound.index)
    X_control = control_data[features]
    y_control = pd.Series(0, index=X_control.index)

    X = pd.concat([X_compound, X_control])
    y = pd.concat([y_compound, y_control])

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train MLP classifier
    mlp_clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
    mlp_clf.fit(X_train_scaled, y_train)

    # Make predictions
    y_pred = mlp_clf.predict(X_test_scaled)
    y_pred_proba = mlp_clf.predict_proba(X_test_scaled)[:, 1]

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    return accuracy, precision, recall, f1, roc_auc, len(compound_data), len(control_data)

# Iterate through all dosages
all_results = {}
for dosage_code, dosage_label in dosage_dict.items():
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Identify control samples
    control_samples = dosage_data[dosage_data['Compound'] == '03']
    
    # Initialize results for this dosage
    results = {}
    
    # Train and evaluate classifiers for each compound at the current dosage
    for compound in dosage_data['Compound'].unique():
        if compound in compound_dict:  # Only process compounds in our dictionary
            compound_data = dosage_data[dosage_data['Compound'] == compound]
            if not compound_data.empty:
                accuracy, precision, recall, f1, roc_auc, n_compound, n_control = train_evaluate_classifier(compound_data, control_samples)
                results[compound] = {
                    'accuracy': accuracy,
                    'precision': precision,
                    'recall': recall,
                    'f1': f1,
                    'roc_auc': roc_auc,
                    'n_compound': n_compound,
                    'n_control': n_control
                }
    
    all_results[dosage_label] = results

    # Plot results for the current dosage
    compounds = list(results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

    print("\nSample sizes per compound at dosage level:", dosage_label)
    print(f"{'Compound':<15}{'Compound Samples':<20}{'Control Samples':<20}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['n_compound']:<20}{metrics_values['n_control']:<20}")

    # Adjust figure size and font sizes
    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Plot metrics
    for metric in metrics:
        values = [results[compound][metric] for compound in compounds]
        ax1.plot(compounds, values, marker='o', label=metric)

    # Axis settings
    ax1.set_xlabel('Compound', fontsize=16)
    ax1.set_ylabel('Metric Value', fontsize=16)
    ax1.tick_params(axis='both', which='major', labelsize=14)
    ax1.set_ylim(0, 1)
    ax1.grid(True)

    # Create a second y-axis for sample sizes
    ax2 = ax1.twinx()

    # Plot sample sizes as bars
    x = np.arange(len(compounds))
    width = 0.35
    control_samples = [results[compound]['n_control'] for compound in compounds]
    compound_samples = [results[compound]['n_compound'] for compound in compounds]

    ax2.bar(x - width/2, control_samples, width, label='Control Samples', alpha=0.5, color='lightblue')
    ax2.bar(x + width/2, compound_samples, width, label='Compound Samples', alpha=0.5, color='lightgreen')

    # Axis settings for second y-axis
    ax2.set_ylabel('Number of samples', fontsize=14)
    ax2.tick_params(axis='y', which='major', labelsize=12)

    # Set x-axis ticks and labels
    plt.xticks(x, [compound_dict[compound] for compound in compounds], fontsize=12)

    # Title and layout adjustments
    plt.title(f'Evaluation Metrics for Control (03) vs Compounds at {dosage_label}', fontsize=16)
    plt.tight_layout()

    # Move legends outside of the graph
    ax1.legend(loc='upper left', bbox_to_anchor=(1.1, 1), fontsize=12)
    ax2.legend(loc='upper left', bbox_to_anchor=(1.1, 0.6), fontsize=12)

    # Show the plot
    plt.show()

    # Print metrics summary
    print("\nMetrics summary:")
    print(f"{'Compound':<15}{'Accuracy':<15}{'Precision':<15}{'Recall':<15}{'F1-Score':<15}{'ROC AUC':<15}")
    for compound, metrics_values in results.items():
        print(f"{compound_dict[compound]:<15}{metrics_values['accuracy']:<15.4f}{metrics_values['precision']:<15.4f}{metrics_values['recall']:<15.4f}{metrics_values['f1']:<15.4f}{metrics_values['roc_auc']:<15.4f}")

In [ ]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample

# Load the data
file_path = r'C:\Users\ak20adb\OneDrive - University of Hertfordshire\Desktop\Celleste 30k test\full datasets\plate_all_zs_new.pkl'
data = pd.read_pickle(file_path)

# Compound dictionary (select compounds for comparison)
compound_dict = {
    '03': 'control',
    '04': 'CuO',
    '05': r'CeO$_2$',  # Use LaTeX for subscript
    '06': r'SiO$_2$',  # Use LaTeX for subscript
    '07': 'ZnO'
}
# Dosage dictionary
dosage_dict = {
    'B': '100μg/ml',
    'C': '33μg/ml',
    'D': '11μg/ml',
    'E': '3μg/ml',
    'F': '1μg/ml',
    'G': '0.4μg/ml'
}

# Features for analysis
features = ['Region:Circularity', 'Region:Clumpiness', 'Region:Diameter, Mean',
            'Region:Fractal Dimension', 'Region:Heterogeneity', 'Region:Roundness',
            'Nuclear Area', 'Percent Area Parent', 'Feature Area',
            'red_intensity', 'green_intensity']

# Extract compound and dosage from Display Name
data['Compound'] = data['Display Name'].str[1:]
data['Dosage'] = data['Display Name'].str[0]

# Convert features to numeric, replacing non-numeric values with NaN
for feature in features:
    data[feature] = pd.to_numeric(data[feature], errors='coerce')

# Remove any rows with NaN values
data = data.dropna()

# Create figure for subplots
n_dosages = len(dosage_dict)
fig = plt.figure(figsize=(14, 4*n_dosages))

# Increase font scale for seaborn heatmap
sns.set(font_scale=1.2)  # Increase overall font size

# Iterate through all dosages
for i, (dosage_code, dosage_label) in enumerate(dosage_dict.items()):
    print(f"\nProcessing dosage: {dosage_label} ({dosage_code})")
    
    # Filter data for the current dosage
    dosage_data = data[data['Dosage'] == dosage_code]
    if dosage_data.empty:
        print(f"No data found for dosage {dosage_label}. Skipping...")
        continue
    
    # Prepare data for all compounds
    all_X = []
    all_y = []
    compound_sizes = {}
    
    # First, collect all data and determine sizes
    for compound_code in compound_dict.keys():
        compound_data = dosage_data[dosage_data['Compound'] == compound_code]
        if not compound_data.empty:
            compound_sizes[compound_code] = len(compound_data)
    
    # Find minimum size for balanced sampling
    if compound_sizes:
        min_size = min(compound_sizes.values())
        
        # Resample data for each compound
        for compound_code in compound_sizes.keys():
            compound_data = dosage_data[dosage_data['Compound'] == compound_code]
            if len(compound_data) > min_size:
                compound_data = resample(compound_data, replace=False, n_samples=min_size, random_state=42)
            elif len(compound_data) < min_size:
                compound_data = resample(compound_data, replace=True, n_samples=min_size, random_state=42)
            
            all_X.append(compound_data[features])
            all_y.extend([compound_dict[compound_code]] * len(compound_data))
    
    # Combine all data
    X = pd.concat(all_X)
    y = pd.Series(all_y)
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Label encode the target variable
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_test_encoded = le.transform(y_test)
    
    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train MLP classifier
    mlp_clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42)
    mlp_clf.fit(X_train_scaled, y_train_encoded)
    
    # Make predictions
    y_pred_encoded = mlp_clf.predict(X_test_scaled)
    
    # Convert predictions back to original labels
    y_pred = le.inverse_transform(y_pred_encoded)
    y_test = le.inverse_transform(y_test_encoded)
    
    # Calculate feature importance
    result = permutation_importance(
        mlp_clf, X_test_scaled, y_test_encoded, 
        n_repeats=10, 
        random_state=42
    )
    
    feature_importance = result.importances_mean
    
    # Create subplots for current dosage
    plt.subplot(n_dosages, 2, 2*i + 1)  # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(compound_dict.values()),
                yticklabels=list(compound_dict.values()))
    plt.title(f'Confusion Matrix - {dosage_label}', fontsize=14)
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('True', fontsize=12)
    
    
    plt.subplot(n_dosages, 2, 2*i + 2)  # Feature Importance
    sorted_idx = np.argsort(feature_importance)
    plt.barh(np.array(features)[sorted_idx], feature_importance[sorted_idx], color='teal')
    plt.title(f'Feature Importance - {dosage_label}', fontsize=14)
    plt.xlabel('Importance Score', fontsize=12)
    plt.tick_params(axis='y', labelsize=10)  # Adjust y-axis tick font size for clarity
    
    # Print classification report
    print(f"\nClassification Report for {dosage_label}:")
    print(classification_report(y_test, y_pred))

plt.tight_layout()
plt.show()